In [ ]:
from __future__ import annotations

import importlib.util
import json
import sys
from pathlib import Path
from urllib.request import urlretrieve

from IPython.display import Markdown, display


# ============================================================
# 0. Optional shared-layer import
# ============================================================

BRANCH = "Atlas"
REPO = "onestardao/WFGY"
BASE_RAW = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/ProblemMap/Atlas/Fixes/official/demos"
SHARED_RAW = f"{BASE_RAW}/shared"

WORKDIR = Path("/content/wfgy_demo2_shared_runtime")
SHARED_DIR = WORKDIR / "shared"
WORKDIR.mkdir(parents=True, exist_ok=True)
SHARED_DIR.mkdir(parents=True, exist_ok=True)


def import_module_from_file(module_name: str, file_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, str(file_path))
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load spec for {module_name} from {file_path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


SHARED_OK = False
display_helpers = None
demo_utils = None
shared_status_note = "Shared helper layer was not loaded. Fallback display mode is active."

try:
    for filename in ("demo_utils.py", "display_helpers.py"):
        urlretrieve(f"{SHARED_RAW}/{filename}", SHARED_DIR / filename)

    demo_utils = import_module_from_file("wfgy_demo_utils", SHARED_DIR / "demo_utils.py")
    display_helpers = import_module_from_file("wfgy_display_helpers", SHARED_DIR / "display_helpers.py")
    SHARED_OK = True
    shared_status_note = "Shared helper layer loaded successfully from the Atlas branch."
except Exception as exc:
    shared_status_note = f"Shared helper layer not loaded. Fallback display mode is active. Reason: {type(exc).__name__}"


# ============================================================
# 1. Fallback display helpers
# ============================================================

def md(text: str):
    display(Markdown(text))


def fallback_section_title(title: str, level: int = 2):
    level = max(1, min(level, 6))
    md(f"{'#' * level} {title}")


def fallback_callout(text: str):
    md(f"> {text}")


def fallback_mode_note(mode_label: str, note: str | None = None):
    lines = [
        "### Mode",
        "",
        f"- **Current mode:** {mode_label}",
    ]
    if note:
        lines.append(f"- **Note:** {note}")
    md("\n".join(lines))


def fallback_json_card(title: str, data):
    md(f"### {title}\n\n```json\n{json.dumps(data, indent=2, ensure_ascii=False)}\n```")


def fallback_route_card(primary_family: str, secondary_family: str, best_current_fit: str, broken_invariant: str, title: str = "Route Summary"):
    md(
        "\n".join(
            [
                f"### {title}",
                "",
                f"- **Primary family:** {primary_family}",
                f"- **Secondary family:** {secondary_family}",
                f"- **Best current fit:** {best_current_fit}",
                f"- **Broken invariant:** {broken_invariant}",
            ]
        )
    )


def fallback_before_after_card(before, after, before_label: str = "Before", after_label: str = "After", title: str = "Before / After"):
    if isinstance(before, (dict, list)):
        before_text = json.dumps(before, indent=2, ensure_ascii=False)
    else:
        before_text = str(before)

    if isinstance(after, (dict, list)):
        after_text = json.dumps(after, indent=2, ensure_ascii=False)
    else:
        after_text = str(after)

    md(
        "\n".join(
            [
                f"### {title}",
                "",
                f"#### {before_label}",
                "",
                "```text",
                before_text,
                "```",
                "",
                f"#### {after_label}",
                "",
                "```text",
                after_text,
                "```",
            ]
        )
    )


def fallback_checklist(title: str, items):
    lines = [f"### {title}", ""]
    for item in items:
        lines.append(f"- [x] {item}")
    md("\n".join(lines))


if not SHARED_OK:
    class FallbackDisplayHelpers:
        display_section_title = staticmethod(fallback_section_title)
        display_callout = staticmethod(fallback_callout)
        display_mode_note = staticmethod(fallback_mode_note)
        display_json_card = staticmethod(fallback_json_card)
        display_route_card = staticmethod(fallback_route_card)
        display_before_after_card = staticmethod(fallback_before_after_card)
        display_checklist = staticmethod(fallback_checklist)

    display_helpers = FallbackDisplayHelpers()


# ============================================================
# 2. Original replay content preserved
# ============================================================

MODE = "replay"

INPUT_CASE = {
    "demo_id": "demo_f5_observability_first",
    "demo_version": "v1",
    "case_id": "f5_observability_case_001",
    "title": "Workflow failure remains too opaque to diagnose correctly",
    "task_type": "workflow_debugging_visibility_case",
    "family_target": {
        "primary_family": "F5",
        "secondary_family": "F4",
        "best_current_fit": "F5_N01 Failure Path Opacity",
        "broken_invariant": "failure_path_visibility_broken"
    },
    "user_question": "Why is the workflow returning irrelevant answers, and what should be fixed first?",
    "thin_trace": [
        "Step 1: User query received",
        "Step 2: Retrieval executed",
        "Step 3: Final answer produced",
        "Observed symptom: answer is irrelevant to the user question"
    ],
    "observability_uplift": {
        "retrieval_trace": [
            "retriever_query = 'general company summary'",
            "top_k = 2",
            "returned_chunk_ids = ['chunk_014', 'chunk_019']",
            "both chunks are broad product overviews, not release-note evidence"
        ],
        "candidate_trace": [
            "candidate_answer_1 = 'The workflow likely needs a stronger generation step'",
            "candidate_answer_2 = 'The retrieval target appears off-topic relative to the question'"
        ],
        "post_check_trace": [
            "answer_to_question_alignment = low",
            "evidence_to_answer_alignment = low",
            "retrieval_to_question_alignment = low"
        ]
    }
}

REPLAY_OUTPUTS = {
    "baseline_diagnosis": "The workflow likely needs a stronger final prompt or a better answer generation step. Try improving the model instructions first.",
    "baseline_problem": "The diagnosis jumps too early to a direct fix even though the workflow is still too opaque for a safe root-cause claim.",
    "repaired_diagnosis": "The first repair move should be observability uplift. The workflow should not be treated as execution-first or reasoning-first yet, because the visible trace is still too thin. Once retrieval trace, candidate trace, and post-check trace are exposed, the system becomes diagnosable and the off-target retrieval signal becomes inspectable.",
    "before_state": "opaque",
    "after_state": "diagnosable",
    "summary": {
        "baseline_outcome": "The final answer is wrong, but the system exposes too little trace structure to determine which stage failed first.",
        "first_repair_move": [
            "observability insertion",
            "trace exposure",
            "diagnostic logging uplift",
            "failure-surface clarification"
        ],
        "final_outcome": "After observability is inserted, the failure path becomes legible enough to show that retrieval selection drifted before downstream intervention should be attempted."
    },
    "route_replay": {
        "why_primary_f5": "The first practical failure is that the operator cannot yet see enough of the failure path to choose a reliable deeper intervention.",
        "why_not_primary_f4": "Execution pressure may exist, but closure repair should not be the first move when the system still lacks stage-level visibility.",
        "teaching_line": "Some failures should be repaired through observability first, because the system is still too opaque to diagnose correctly."
    },
    "before_after_comparison": {
        "before": {
            "answer": "Lite includes those features.",
            "trace_state": "opaque",
            "repair_state": "unrepaired",
            "operator_position": "can_see_failure_but_not_failure_path"
        },
        "after": {
            "answer": "Lite includes those features.",
            "trace_state": "legible_enough_for_targeted_next_move",
            "repair_state": "observability_repaired",
            "operator_position": "can_identify_retrieval_drift_as_the_next_target"
        },
        "what_changed": [
            "The first improvement is not yet the final answer. The first improvement is visibility.",
            "The operator moved from guessing about failure causes to seeing a plausible failure path.",
            "The correct next repair can now be chosen with more confidence."
        ]
    },
    "improved_visibility_snapshot": {
        "new_visibility_value": [
            "The operator can now see that retrieval drift happened first.",
            "The operator can now see that the post-check failed to reject the wrong anchor.",
            "The system is now diagnosable enough to choose the next repair layer with less guesswork."
        ]
    }
}

EXPECTED_OUTPUT = {
    "primary_family": "F5",
    "secondary_family": "F4",
    "best_current_fit": "F5_N01 Failure Path Opacity",
    "broken_invariant": "failure_path_visibility_broken",
    "first_repair_move": "observability insertion",
    "minimum_success_contract": {
        "primary_family": "F5",
        "secondary_family": "F4",
        "best_current_fit": "F5_N01 Failure Path Opacity",
        "broken_invariant": "failure_path_visibility_broken",
        "fit_level": "node level",
        "confidence": "medium or higher",
        "evidence_sufficiency": "sufficient_for_diagnosability_first_cut"
    }
}

baseline_prompt = f"""
You are a workflow debugging assistant.

A system received a user query, ran retrieval, and produced a final answer.
The final answer was irrelevant to the user question.

Available trace:
- {INPUT_CASE["thin_trace"][0]}
- {INPUT_CASE["thin_trace"][1]}
- {INPUT_CASE["thin_trace"][2]}
- {INPUT_CASE["thin_trace"][3]}

Explain what probably went wrong.
Then recommend one direct fix to apply immediately.

Assume the current trace is enough.
Keep the answer short and confident.
""".strip()

repaired_prompt = f"""
You are diagnosing a workflow failure.

Question:
{INPUT_CASE["user_question"]}

Thin trace:
- {INPUT_CASE["thin_trace"][0]}
- {INPUT_CASE["thin_trace"][1]}
- {INPUT_CASE["thin_trace"][2]}
- {INPUT_CASE["thin_trace"][3]}

Additional observability:
Retrieval trace:
- {INPUT_CASE["observability_uplift"]["retrieval_trace"][0]}
- {INPUT_CASE["observability_uplift"]["retrieval_trace"][1]}
- {INPUT_CASE["observability_uplift"]["retrieval_trace"][2]}
- {INPUT_CASE["observability_uplift"]["retrieval_trace"][3]}

Candidate trace:
- {INPUT_CASE["observability_uplift"]["candidate_trace"][0]}
- {INPUT_CASE["observability_uplift"]["candidate_trace"][1]}

Post-check trace:
- {INPUT_CASE["observability_uplift"]["post_check_trace"][0]}
- {INPUT_CASE["observability_uplift"]["post_check_trace"][1]}
- {INPUT_CASE["observability_uplift"]["post_check_trace"][2]}

Answer in this order:
1. What the first failure family is
2. Why F5 is a better first cut than F4
3. What the first repair move should be
4. What the visible evidence now suggests
""".strip()


# ============================================================
# 3. Render improved replay notebook
# ============================================================

display_helpers.display_section_title("Demo 2 · F5 Observability First", level=1)
display_helpers.display_callout(
    "Replay-only MVP notebook. This version preserves the original replay teaching content and adds a clearer route summary, stronger before / after presentation, and a cleaner official display flow."
)
display_helpers.display_mode_note(
    "Replay-only MVP",
    "No API key required. The first goal is to show that observability uplift changes the repair path before deeper workflow intervention should begin."
)

display_helpers.display_json_card(
    "Case Overview",
    {
        "demo_id": INPUT_CASE["demo_id"],
        "demo_version": INPUT_CASE["demo_version"],
        "case_id": INPUT_CASE["case_id"],
        "title": INPUT_CASE["title"],
        "task_type": INPUT_CASE["task_type"],
        "user_question": INPUT_CASE["user_question"],
    },
)

display_helpers.display_route_card(
    primary_family=INPUT_CASE["family_target"]["primary_family"],
    secondary_family=INPUT_CASE["family_target"]["secondary_family"],
    best_current_fit=INPUT_CASE["family_target"]["best_current_fit"],
    broken_invariant=INPUT_CASE["family_target"]["broken_invariant"],
    title="Atlas Route Summary",
)

display_helpers.display_json_card("Thin Trace", INPUT_CASE["thin_trace"])
display_helpers.display_json_card("Observability Uplift", INPUT_CASE["observability_uplift"])

display_helpers.display_json_card(
    "Original Baseline Prompt",
    {"baseline_prompt": baseline_prompt},
)

display_helpers.display_json_card(
    "Original Repaired Prompt",
    {"repaired_prompt": repaired_prompt},
)

display_helpers.display_json_card(
    "Replay Summary",
    REPLAY_OUTPUTS["summary"],
)

display_helpers.display_json_card(
    "Why F5 Is Primary",
    REPLAY_OUTPUTS["route_replay"],
)

display_helpers.display_json_card(
    "Baseline Replay Diagnosis",
    {
        "baseline_diagnosis": REPLAY_OUTPUTS["baseline_diagnosis"],
        "baseline_problem": REPLAY_OUTPUTS["baseline_problem"],
    },
)

display_helpers.display_json_card(
    "Repaired Replay Diagnosis",
    {
        "repaired_diagnosis": REPLAY_OUTPUTS["repaired_diagnosis"],
    },
)

display_helpers.display_before_after_card(
    before=REPLAY_OUTPUTS["before_after_comparison"]["before"],
    after=REPLAY_OUTPUTS["before_after_comparison"]["after"],
    before_label="Before",
    after_label="After",
    title="Replay Before / After",
)

display_helpers.display_json_card(
    "What Changed",
    {
        "what_changed": REPLAY_OUTPUTS["before_after_comparison"]["what_changed"],
        "new_visibility_value": REPLAY_OUTPUTS["improved_visibility_snapshot"]["new_visibility_value"],
    },
)

display_helpers.display_json_card(
    "Expected Minimum Success Contract",
    EXPECTED_OUTPUT["minimum_success_contract"],
)

display_helpers.display_checklist(
    "Quick Validation Checklist",
    [
        "The notebook clearly presents Demo 2 as replay-only MVP",
        "The original replay teaching content remains preserved",
        "The atlas route summary is visible",
        "The baseline and repaired replay diagnoses are both visible",
        "The before / after visibility shift is easy to inspect",
        "The minimum success contract is visible",
        "The notebook is readable without any API key",
    ],
)

display_helpers.display_json_card(
    "Shared Layer Status",
    {
        "shared_layer_loaded": SHARED_OK,
        "note": shared_status_note,
        "working_directory": str(WORKDIR),
    },
)

display_helpers.display_callout(
    "Main lesson: the first useful repair is not yet a full system fix. The first useful repair is visibility uplift, because the workflow is still too opaque for a safe deeper cut."
)

md(
    """
## Back to the main page

Read the full product page here:
[Problem Map 3.0 Troubleshooting Atlas](https://github.com/onestardao/WFGY/blob/main/ProblemMap/wfgy-ai-problem-map-troubleshooting-atlas.md)

If you like the project, star the repo ⭐
"""
)